# 01 — Build the 12-task pilot manifest

This extracts the selected benchmark rows. It copies the question and supplied data, but never puts the existing model answer into Dexter's prompt.

In [ ]:
from pathlib import Path
import json, os
import openpyxl

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'src' / 'state_management').is_dir() else WORKING_DIR.parent
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
RUNS_DIR = EXPERIMENTS_DIR / 'runs'
RUNS_DIR.mkdir(exist_ok=True)
DATASET_PATH = Path(os.environ.get('DATASET_PATH', Path.home() / 'Downloads' / 'dataset1.xlsx'))
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')
print(f'Using dataset: {DATASET_PATH}')

In [ ]:
# Keep this list fixed across every condition.
PILOT_EPISODES = [
    ('plan_entity', [(5, 'groww-01'), (49, 'groww-02'), (76, 'groww-03')]),
    ('metric_label', [(29, 'motilal-01'), (33, 'motilal-02'), (94, 'motilal-03')]),
    ('plan_ambiguity', [(83, 'nippon-01'), (84, 'nippon-02'), (16, 'nippon-03')]),
    ('stale_data', [(17, 'hdfc-01'), (98, 'hdfc-02'), (31, 'hdfc-03')]),
]
LESSONS = {
    'plan_entity': 'Keep Groww the app distinct from Groww Largecap Fund; label plan and expense ratio.',
    'metric_label': 'Distinguish total return from CAGR; preserve the supplied metric label and riskometer requirement.',
    'plan_ambiguity': 'Do not silently assume Direct plan when wording is ambiguous; ask or flag the assumption.',
    'stale_data': 'Use only the supplied data snapshot; never claim an invented live NAV.',
}
workbook = openpyxl.load_workbook(DATASET_PATH, read_only=True, data_only=True)
sheet = workbook['Review']
HEADERS = {sheet.cell(1, column).value: column for column in range(1, sheet.max_column + 1)}
required = ['Item', 'Question asked', 'What the question means (intended reading)', 'Data the model was given', 'What the data figure actually measures']
missing = [header for header in required if header not in HEADERS]
if missing:
    raise ValueError(f'Unexpected Review sheet; missing columns: {missing}')
HEADERS

In [ ]:
manifest = []
for episode, rows in PILOT_EPISODES:
    for turn, (row_number, task_id) in enumerate(rows, start=1):
        row = {name: sheet.cell(row_number + 1, column).value for name, column in HEADERS.items()}
        if row['Item'] != row_number:
            raise ValueError(f'Expected benchmark item {row_number}, found {row["Item"]}')
        manifest.append({
            'task_id': task_id, 'source_row': row_number, 'episode': episode, 'turn': turn,
            'question': row['Question asked'],
            'provided_data': row['Data the model was given'],
            'intended_reading': row['What the question means (intended reading)'],
            'figure_meaning': row['What the data figure actually measures'],
            'lesson_target': LESSONS[episode],
            'success_rule': 'Answer the intended reading and correctly label every number. Flag or ask when under-specified.',
        })
assert len(manifest) == 12
assert len({task['task_id'] for task in manifest}) == 12
[(task['task_id'], task['source_row'], task['episode']) for task in manifest]

In [ ]:
MANIFEST_PATH = RUNS_DIR / 'tasks.jsonl'
with MANIFEST_PATH.open('w', encoding='utf-8') as handle:
    for task in manifest:
        handle.write(json.dumps(task, ensure_ascii=False) + '\n')
print(f'Wrote {len(manifest)} tasks to {MANIFEST_PATH}')
print('Prompt template: Question: {question}\n\nData supplied for this task only:\n{provided_data}')

In [ ]:
# Preview prompt inputs. Do NOT add the existing Model answer column.
for task in manifest:
    print(f"{task['task_id']} | row {task['source_row']} | {task['question']}")